# VLM Fine-Tuning: LoRA vs AdaLoRA on DocVQA

**Model**: SmolVLM-256M  
**Task**: Visual Question Answering (VQA)  
**Dataset**: DocVQA  
**Research Question**: Does AdaLoRA improve parameter-efficient fine-tuning of SmolVLM-256M compared with standard LoRA under a similar parameter budget?

## 1 — Install Dependencies

In [ ]:
!pip install torch torchvision transformers peft datasets accelerate Pillow evaluate pandas matplotlib tqdm scikit-learn -q

## 2 — Imports & Reproducibility

In [ ]:
import os, sys, random, json, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch
from tqdm import tqdm
from PIL import Image
from datasets import load_dataset
from transformers import (
    AutoProcessor,
    AutoModelForVision2Seq,
    get_linear_schedule_with_warmup,
)
from peft import (
    LoraConfig, AdaLoraConfig,
    get_peft_model, PeftModel, TaskType,
)
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

## 3 — Load DocVQA

In [ ]:
TRAIN_SIZE, VAL_SIZE, TEST_SIZE = 2000, 500, 500

ds = load_dataset('HuggingFaceM4/DocumentVQA', trust_remote_code=True)
train_hf = ds['train'].shuffle(seed=SEED).select(range(TRAIN_SIZE))
val_hf   = ds['validation'].shuffle(seed=SEED).select(range(VAL_SIZE))
test_hf  = ds['test'].shuffle(seed=SEED).select(range(min(TEST_SIZE, len(ds['test']))))

print('Train:', len(train_hf), '| Val:', len(val_hf), '| Test:', len(test_hf))

# Quick preview
sample = train_hf[0]
print('Question:', sample['question'])
print('Answer  :', sample['answers'][0])
sample['image']

## 4 — Dataset Class

In [ ]:
class DocVQADataset(Dataset):
    def __init__(self, hf_dataset, processor, max_length=256):
        self.data, self.processor, self.max_length = hf_dataset, processor, max_length

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item     = self.data[idx]
        image    = item['image']
        question = item['question']
        answers  = item.get('answers') or item.get('answer') or ['']
        answer   = answers[0] if isinstance(answers, list) else answers
        prompt   = f'Question: {question}\nAnswer:'

        enc = self.processor(
            images=image, text=prompt,
            return_tensors='pt', padding='max_length',
            truncation=True, max_length=self.max_length)
        lbl = self.processor.tokenizer(
            answer, return_tensors='pt',
            padding='max_length', truncation=True, max_length=64)
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'pixel_values':   enc['pixel_values'].squeeze(0),
            'labels':         lbl['input_ids'].squeeze(0),
            'answer_text':    answer,
            'question_text':  question,
        }

def collate_fn(batch):
    keys = ['input_ids','attention_mask','pixel_values','labels']
    out  = {k: torch.stack([b[k] for b in batch]) for k in keys}
    out['answer_text']   = [b['answer_text']   for b in batch]
    out['question_text'] = [b['question_text'] for b in batch]
    return out

## 5 — Load SmolVLM-256M

In [ ]:
MODEL_ID  = 'HuggingFaceTB/SmolVLM-256M-Instruct'
DTYPE     = torch.bfloat16 if torch.cuda.is_available() else torch.float32
processor = AutoProcessor.from_pretrained(MODEL_ID)

def fresh_model():
    m = AutoModelForVision2Seq.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map='auto')
    m.config.use_cache = False
    return m

print('Processor loaded.')

## 6 — LoRA Configuration

In [ ]:
TARGET_MODULES = ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj']
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05

def build_lora():
    base = fresh_model()
    cfg  = LoraConfig(
        task_type=TaskType.CAUSAL_LM, r=LORA_R,
        lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=TARGET_MODULES, bias='none')
    model = get_peft_model(base, cfg)
    model.print_trainable_parameters()
    return model

lora_model = build_lora()

## 7 — AdaLoRA Configuration

In [ ]:
ADA_INIT_R, ADA_TARGET_R = 32, 16

def build_adalora():
    base = fresh_model()
    cfg  = AdaLoraConfig(
        task_type=TaskType.CAUSAL_LM, init_r=ADA_INIT_R,
        target_r=ADA_TARGET_R, lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT, target_modules=TARGET_MODULES,
        deltaT=10, beta1=0.85, beta2=0.85, orth_reg_weight=0.5)
    model = get_peft_model(base, cfg)
    model.print_trainable_parameters()
    return model

adalora_model = build_adalora()

## 8 — Training Helper

In [ ]:
BATCH_SIZE    = 4
LR            = 2e-4
NUM_EPOCHS    = 3
WARMUP_RATIO  = 0.1
MAX_GRAD_NORM = 1.0

def train_model(model, name='lora'):
    train_ds = DocVQADataset(train_hf, processor)
    val_ds   = DocVQADataset(val_hf,   processor)
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=0.01)
    total_steps = NUM_EPOCHS * len(train_dl)
    scheduler   = get_linear_schedule_with_warmup(
        optimizer, int(WARMUP_RATIO * total_steps), total_steps)

    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train(); t_loss = 0.0
        for step, batch in enumerate(tqdm(train_dl, desc=f'[{name}] Epoch {epoch} train')):
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            pix  = batch['pixel_values'].to(DEVICE)
            lbl  = batch['labels'].to(DEVICE); lbl[lbl == 0] = -100
            out  = model(input_ids=ids, attention_mask=mask, pixel_values=pix, labels=lbl)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
            if hasattr(model, 'update_and_allocate'): model.update_and_allocate(step)
            t_loss += out.loss.item()

        model.eval(); v_loss = 0.0
        with torch.no_grad():
            for batch in tqdm(val_dl, desc=f'[{name}] Epoch {epoch} val'):
                ids  = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                pix  = batch['pixel_values'].to(DEVICE)
                lbl  = batch['labels'].to(DEVICE); lbl[lbl == 0] = -100
                v_loss += model(
                    input_ids=ids, attention_mask=mask,
                    pixel_values=pix, labels=lbl).loss.item()

        tl = t_loss / len(train_dl); vl = v_loss / len(val_dl)
        history['train_loss'].append(tl); history['val_loss'].append(vl)
        print(f'  Epoch {epoch} | train_loss={tl:.4f} | val_loss={vl:.4f}')

        ckpt = f'checkpoints/{name}'
        os.makedirs(ckpt, exist_ok=True)
        model.save_pretrained(ckpt); processor.save_pretrained(ckpt)

    return history

## 9 — Train LoRA Baseline

In [ ]:
lora_history = train_model(lora_model, name='lora')

## 10 — Train AdaLoRA

In [ ]:
adalora_history = train_model(adalora_model, name='adalora')

## 11 — Evaluation Helper

In [ ]:
def normalise(text):
    text = text.lower().strip()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return ' '.join(text.split())

def classify_question(q):
    q = q.lower()
    if re.search(r'how many|how much|number|amount|total|count|year|date|price|cost', q):
        return 'Numerical'
    if re.search(r'where|location|position|address|table|column|row|section', q):
        return 'Layout'
    return 'Text/OCR'

@torch.no_grad()
def evaluate(model, hf_data):
    model.eval(); records = []
    for item in tqdm(hf_data, desc='Evaluating'):
        image    = item['image']
        question = item['question']
        answers  = item.get('answers') or item.get('answer') or ['']
        gt       = answers[0] if isinstance(answers, list) else answers
        prompt   = f'Question: {question}\nAnswer:'
        enc      = processor(
            images=image, text=prompt, return_tensors='pt',
            padding=True, truncation=True, max_length=256).to(DEVICE)
        gen_ids  = model.generate(**enc, max_new_tokens=64, do_sample=False)
        gen_ids  = gen_ids[:, enc['input_ids'].shape[-1]:]
        pred     = processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()
        score    = float(normalise(pred) == normalise(gt))
        records.append({'question': question, 'ground_truth': gt,
                        'prediction': pred, 'score': score,
                        'q_type': classify_question(question)})
    return pd.DataFrame(records)

## 12 — Evaluate LoRA

In [ ]:
lora_df = evaluate(lora_model, test_hf)
print('LoRA overall VQA score:', round(lora_df['score'].mean(), 4))

## 13 — Evaluate AdaLoRA

In [ ]:
adalora_df = evaluate(adalora_model, test_hf)
print('AdaLoRA overall VQA score:', round(adalora_df['score'].mean(), 4))

## 14 — Results Table

In [ ]:
rows = []
for qt in ['Text/OCR', 'Numerical', 'Layout']:
    ls  = lora_df[lora_df['q_type'] == qt]['score'].mean()
    als = adalora_df[adalora_df['q_type'] == qt]['score'].mean()
    rows.append({'Question Type': qt,
                 'LoRA': round(ls, 4), 'AdaLoRA': round(als, 4),
                 'Difference': round(als - ls, 4)})
rows.append({'Question Type': 'Overall',
             'LoRA':    round(lora_df['score'].mean(), 4),
             'AdaLoRA': round(adalora_df['score'].mean(), 4),
             'Difference': round(adalora_df['score'].mean() - lora_df['score'].mean(), 4)})
results_table = pd.DataFrame(rows)
os.makedirs('results', exist_ok=True)
results_table.to_csv('results/results.csv', index=False)
results_table

## 15 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
epochs = range(1, NUM_EPOCHS + 1)

ax = axes[0]
ax.plot(epochs, lora_history['train_loss'],    'b-o',  label='LoRA train')
ax.plot(epochs, lora_history['val_loss'],      'b--o', label='LoRA val')
ax.plot(epochs, adalora_history['train_loss'], 'r-s',  label='AdaLoRA train')
ax.plot(epochs, adalora_history['val_loss'],   'r--s', label='AdaLoRA val')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Training and Validation Loss'); ax.legend()

ax = axes[1]
q_types = ['Text/OCR', 'Numerical', 'Layout', 'Overall']
x = np.arange(len(q_types)); w = 0.35
ax.bar(x - w/2,
    [results_table.loc[results_table['Question Type'] == qt, 'LoRA'].values[0]    for qt in q_types],
    w, label='LoRA')
ax.bar(x + w/2,
    [results_table.loc[results_table['Question Type'] == qt, 'AdaLoRA'].values[0] for qt in q_types],
    w, label='AdaLoRA')
ax.set_xticks(x); ax.set_xticklabels(q_types, rotation=15)
ax.set_ylabel('VQA Score'); ax.set_title('VQA Score by Question Type'); ax.legend()

plt.tight_layout()
plt.savefig('results/training_curves.png', dpi=150)
plt.show()
print('Saved -> results/training_curves.png')

## 16 — AdaLoRA Rank Analysis

In [ ]:
ranks = {}
for name, param in adalora_model.named_parameters():
    if 'lora_E' in name and param.requires_grad:
        eff = (param.abs() > 1e-6).sum().item()
        clean = name.replace('.lora_E', '').replace('base_model.model.', '')
        ranks[clean] = int(eff)

print('Total AdaLoRA modules:', len(ranks))
rank_df = pd.DataFrame(ranks.items(), columns=['Module', 'Effective Rank'])
rank_df = rank_df.sort_values('Effective Rank', ascending=False)
print(rank_df.head(15).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
modules = list(ranks.keys())[:30]
values  = [ranks[m] for m in modules]
ax.bar(range(len(modules)), values, color='steelblue', edgecolor='black', linewidth=0.5)
ax.set_xticks(range(len(modules)))
ax.set_xticklabels(modules, rotation=90, fontsize=7)
ax.set_ylabel('Effective Rank')
ax.set_title('AdaLoRA - Final Rank Allocation per Module')
plt.tight_layout()
plt.savefig('results/rank_analysis.png', dpi=150)
plt.show()
print('Saved -> results/rank_analysis.png')

## 17 — Experiment Summary

In [ ]:
print('=' * 60)
print('  EXPERIMENT SUMMARY')
print('=' * 60)
print('  Model   : SmolVLM-256M')
print('  Dataset : DocVQA', TRAIN_SIZE, 'train /', VAL_SIZE, 'val /', TEST_SIZE, 'test')
print('  Epochs  :', NUM_EPOCHS, ' | Batch size:', BATCH_SIZE, ' | LR:', LR)
print('  SEED    :', SEED)
print()
print(results_table.to_string(index=False))
print()
diff = results_table.loc[results_table['Question Type'] == 'Overall', 'Difference'].values[0]
if diff > 0:
    print('CONCLUSION: AdaLoRA outperforms LoRA by', round(diff, 4), '(supports hypothesis)')
elif diff < 0:
    print('CONCLUSION: LoRA outperforms AdaLoRA by', round(abs(diff), 4))
else:
    print('CONCLUSION: Both methods perform equally')